# Localization: Plot Builder

Interactive plot builder for localization experiments.
Generates matplotlib previews and copy-pasteable LaTeX
(pgfplots) source for the paper.

## Metrics

- **accuracy_primary**: the model picks which sentence was
  perturbed, using the first-token logit for each label
  (e.g. logit of " A" vs logit of " B"). Correct if
  the highest-logit label matches the ground truth.
- **accuracy_aggregate**: same idea, but uses the summed
  probability across all token variants of each label.
- **content_accuracy_primary**: for controls only. The model
  picks which sentence matches a topic (e.g. "about animals").
  Uses primary (first-token) logits.

## Error bands

All experiments are binary classification ($n$ Bernoulli
trials). The error band can toggle between:

- **SE** (standard error): $\sqrt{\hat{p}(1-\hat{p})/n}$.
  Shows uncertainty of the accuracy estimate.
- **SD** (standard deviation): $\sqrt{\hat{p}(1-\hat{p})}$.
  Shows the spread of individual binary outcomes. This band
  is much wider (it does not shrink with $n$).

In [ ]:
import os
import pathlib

_this_dir = pathlib.Path(os.path.abspath("")).resolve()
if _this_dir.name == "paper":
    os.chdir(_this_dir.parent)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from wandb_cache import load_sweep

In [ ]:
# Sweeps: the manifest sweeps.yaml (project llm-dropout-noise), loaded in cell 4 from the local cache.
COL_NOISE_GRID = (
    "noise_grid"  # "coarse" (base grid, plus *_ext) | "fine" (zoom sweeps *_fine*)
)

# --- Columns ---
COL_MODEL = "model"
COL_DROPOUT = "perturbation.dropout_rate"
COL_NOISE = "perturbation.noise_std"
COL_SENTENCES = "experiment.sentences_file"
COL_PROMPT = "prompt_turns"

# --- Models ---
MODELS = [
    "llama3_8b",
    "qwen3_14b",
    "qwen3_32b",
    "olmo3_32b",
    "qwen3_8_27b",
    "seed_oss_36b",
    "mistral_small_24b",
    "gemma4_31b",
]
# Olmo3.1-32B training stages (olmo3_32b is the RLVR stage): off by default in the panels.
STAGES = ["olmo3_32b_instruct_base", "olmo3_32b_instruct_sft", "olmo3_32b_instruct_dpo"]
ALL_MODELS = MODELS + STAGES

MODEL_COLORS_MPL = {
    "llama3_8b": "#f4b6c2",
    "qwen3_14b": "#8fae82",
    "qwen3_32b": "#1b3a6b",
    "olmo3_32b": "#8b1a1a",
    "qwen3_8_27b": "#00a3a3",
    "seed_oss_36b": "#b5651d",
    "mistral_small_24b": "#d4a017",
    "gemma4_31b": "#4b86b4",
    "olmo3_32b_instruct_base": "#f0a8a8",
    "olmo3_32b_instruct_sft": "#e06666",
    "olmo3_32b_instruct_dpo": "#c03030",
}
MODEL_COLORS_LATEX = {
    "llama3_8b": "red!40",
    "qwen3_14b": "green!40!black",
    "qwen3_32b": "blue!70!black",
    "olmo3_32b": "red!70!black",
    "qwen3_8_27b": "teal!70!black",
    "seed_oss_36b": "brown!80!black",
    "mistral_small_24b": "yellow!60!black",
    "gemma4_31b": "cyan!60!black",
    "olmo3_32b_instruct_base": "red!25",
    "olmo3_32b_instruct_sft": "red!50",
    "olmo3_32b_instruct_dpo": "red!80",
}

# LaTeX model names: the paper defines \model{<key>} (math_commands.tex). The second entry is the display name,
# used to sort the legends.
MODEL_MACROS_LATEX = {
    "llama3_8b": (r"\model{llama8}", "Llama3.1-8B"),
    "qwen3_14b": (r"\model{qwen14}", "Qwen3-14B"),
    "qwen3_32b": (r"\model{qwen32}", "Qwen3-32B"),
    "olmo3_32b": (r"\model{olmo32}", "Olmo3.1-32B"),
    "qwen3_8_27b": (r"\model{qwen27}", "Qwen3.8-27B"),
    "seed_oss_36b": (r"\model{seed36}", "Seed-OSS-36B"),
    "mistral_small_24b": (r"\model{mistral24}", "Mistral-Small-24B"),
    "gemma4_31b": (r"\model{gemma31}", "Gemma4-31B"),
    "olmo3_32b_instruct_base": ("Olmo3.1-32B Base", "Olmo3.1-32B Base"),
    "olmo3_32b_instruct_sft": ("Olmo3.1-32B SFT", "Olmo3.1-32B SFT"),
    "olmo3_32b_instruct_dpo": ("Olmo3.1-32B DPO", "Olmo3.1-32B DPO"),
}
PAPER_CAPTIONS = {
    "Main": (
        r"\small We present the models with two sentences, perturb the activations of only one, and ask the "
        r"model which sentence is perturbed. Whether the perturbation is \textbf{dropout} (left) or "
        r"\textbf{Gaussian noise} (right), all models can easily detect and locate it (in contrast with the "
        r"chance-level accuracy when neither is applied). Standard errors are below $1.58\%$ and hence "
        r"difficult to visualize.",
        "loc_main_15tok",
    ),
}
DROPOUT_XLABEL = {"mpl": r"Dropout rate $p$", "latex": "Dropout rate $p$"}
NOISE_XLABEL = {"mpl": r"Noise SD $\sigma$", "latex": "Noise SD $\\sigma$"}

METRICS = [
    "accuracy_primary",
    "accuracy_aggregate",
    "accuracy_constrained_primary",
    "accuracy_constrained_aggregate",
    "content_accuracy_primary",
    "content_accuracy_aggregate",
    "content_accuracy_constrained_primary",
    "content_accuracy_constrained_aggregate",
    "roc_auc_constrained_primary",
    "roc_auc_constrained_aggregate",
    "macro_f1_constrained_primary",
    "macro_f1_constrained_aggregate",
    "entropy_mean",
    "argmax_is_label_rate",  # top vocab token is a label (A/B) vs anything else
    "position_bias",  # mean probability mass on the first label (A)
]

In [ ]:
# --- Load the manifest sweeps from the local cache ---
_manifest = yaml.safe_load(
    (_this_dir / "sweeps.yaml").read_text()
)  # cell 1 defines _this_dir


def _load_key(key):
    """Base sweep `key` of every model plus its zoom sweeps `key_<suffix>` (e.g. loc_noise_fine,
    ctrl_noise_ext); zoom sweeps named *_fine* are tagged "fine", everything else "coarse"."""
    frames = []
    for model, ids in _manifest.items():
        if not isinstance(ids, dict):
            continue
        for k, sid in ids.items():
            if k == key or k.startswith(key + "_"):
                _df = load_sweep(sid, _manifest["project"], require_version=2)
                _df[COL_NOISE_GRID] = "fine" if "_fine" in k else "coarse"
                frames.append(_df)
    return pd.concat(frames, ignore_index=True)


df_main_d, df_main_n = _load_key("loc_dropout"), _load_key("loc_noise")
df_ctrl_d, df_ctrl_n = _load_key("ctrl_dropout"), _load_key("ctrl_noise")
print(
    f"data: main {len(df_main_d)}+{len(df_main_n)} runs, control {len(df_ctrl_d)}+{len(df_ctrl_n)} runs, models {sorted(df_main_d[COL_MODEL].unique())}"
)

# Samples per run
N_PER_RUN = (
    int(df_main_d["total_samples"].mode().iloc[0])
    if "total_samples" in df_main_d.columns
    else 1000
)
print(f"\nSamples per run: {N_PER_RUN}")


# Discover grouping values
def _stem(path):
    return pathlib.PurePosixPath(path).stem


sent_files = sorted(df_main_d[COL_SENTENCES].unique())
sent_map = {_stem(sf): sf for sf in sent_files}
print(f"Sentence lengths: {list(sent_map.keys())}")

prompt_files = sorted(df_ctrl_d[COL_PROMPT].unique())
prompt_map = {p.split("/")[-1]: p for p in prompt_files}
print(f"Control prompts: {list(prompt_map.keys())}")

avail_main = [m for m in METRICS if m in df_main_d.columns]
# Control panel: the 95% criterion is on content accuracy (paper rule = content_accuracy_primary),
# so content metrics come first; detection accuracy on control prompts is a different quantity.
CTRL_METRICS = [m for m in METRICS if m.startswith("content_")] + [
    m for m in METRICS if not m.startswith("content_")
]
avail_ctrl = [m for m in CTRL_METRICS if m in df_ctrl_d.columns]
unavail_ctrl = [m for m in CTRL_METRICS if m not in df_ctrl_d.columns]
print(f"Metrics (main): {avail_main}")
print(f"Metrics (ctrl): {avail_ctrl}")
print(
    f"Unavailable in the loaded control runs (not recorded, cannot be recovered): {unavail_ctrl}"
)

In [ ]:
# ── Curve computation ────────────────────────────────────────


def compute_curves(df, x_col, metric, group_col, group_value, n_per_run, models):
    """Return {model: (x, y_pct, se_pct, sd_pct)}.

    group_value == "all" pools across all values of group_col.
    For accuracy-family metrics, SE and SD come from the
    Bernoulli formula.  For others, they are empirical across
    runs in the group.
    """
    is_pct = "accuracy" in metric or "f1" in metric
    scale = 100 if is_pct else 1
    curves = {}
    for model in models:
        sub = df[df[COL_MODEL] == model]
        if metric not in sub.columns or sub.empty:
            continue
        if group_value != "all":
            sub = sub[sub[group_col] == group_value]
        if sub.empty:
            continue

        grp = sub.groupby(x_col)[metric]
        avg = grp.mean().sort_index() * scale
        if avg.empty:
            continue

        if is_pct:
            p = avg / 100
            pq = p * (1 - p)
            n_total = grp.count() * n_per_run
            se = np.sqrt(pq / n_total) * 100
            sd = np.sqrt(pq) * 100
        else:
            se = grp.sem().sort_index() * scale
            sd = grp.std().sort_index() * scale

        curves[model] = (avg.index.values, avg.values, se.values, sd.values)
    return curves


def clip_curves(curves, lo, hi):
    """Keep only the points with lo <= x <= hi (None = unbounded); drop models left empty."""
    out = {}
    for m, (x, y, se, sd) in curves.items():
        keep = np.ones(len(x), dtype=bool)
        if lo is not None:
            keep &= x >= lo
        if hi is not None:
            keep &= x <= hi
        if keep.any():
            out[m] = (x[keep], y[keep], se[keep], sd[keep])
    return out


# ── Matplotlib ───────────────────────────────────────────────


def plot_panel(
    ax, curves, x_label, metric, error_type, ymin, ymax, is_right=False, ref_lines=None
):
    """ref_lines: list of y-values for gray dashed reference lines."""
    is_pct = "accuracy" in metric or "f1" in metric

    for model in ALL_MODELS:
        if model not in curves:
            continue
        x, y, se, sd = curves[model]
        err = se if error_type == "SE" else sd
        c = MODEL_COLORS_MPL[model]
        ax.plot(x, y, marker="o", markersize=3, linewidth=2, color=c, label=model)
        ax.fill_between(x, y - err, y + err, alpha=0.3, color=c)

    if is_pct:
        ax.axhline(50, color="gray", ls="--", alpha=0.5, label="chance")
    for rl in ref_lines or []:
        ax.axhline(rl, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel(x_label)
    if is_right:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Accuracy (%)" if is_pct else metric)
    ax.grid(True, alpha=0.3)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


# ── LaTeX / pgfplots ────────────────────────────────────────


def latex_panel(
    curves,
    x_label,
    metric,
    error_type,
    models,
    is_right=False,
    ref_lines=None,
    ymin=None,
    ymax=None,
    xlim=(None, None),
):
    is_pct = "accuracy" in metric or "f1" in metric
    all_x = (
        np.concatenate([c[0] for c in curves.values()]) if curves else np.array([0, 1])
    )
    xmin, xmax = float(all_x.min()), float(all_x.max())
    xmin = (
        xmin if xlim[0] is None else float(xlim[0])
    )  # explicit axis range (noise sigma min/max)
    xmax = xmax if xlim[1] is None else float(xlim[1])

    def _fx(v):  # 0.0, 0.98, 0.5 as in the paper
        return f"{v:.1f}" if float(v).is_integer() else f"{v:g}"

    L = []
    L.append(r"\begin{tikzpicture}")
    L.append(r"\begin{axis}[")
    L.append("    height=5cm,")
    L.append("    width=\\linewidth,")
    L.append("    grid=major,")
    L.append(f"    xlabel={{\\small {x_label}}},")
    if is_right:
        L.append("    ylabel={},")
        L.append("    yticklabels={},")
    else:
        ylabel = "Accuracy (\\%)" if is_pct else metric
        L.append(f"    ylabel={{\\small {ylabel}}},")
    L.append(f"    xmin={_fx(xmin)},")
    L.append(f"    xmax={_fx(xmax)},")
    if ymin is None and ymax is None and is_pct:
        ymin, ymax = 0.0, 100.0
    if ymin is not None and ymax is not None:
        L.append(f"    ymin={float(ymin)}, ymax={float(ymax)},")
    L.append("    legend entries={},")
    L.append("]")

    for model in models:
        if model not in curves:
            continue
        x, y, se, sd = curves[model]
        err = se if error_type == "SE" else sd
        c = MODEL_COLORS_LATEX[model]

        up = " ".join(f"({xi:.4f},{yi + ei:.2f})" for xi, yi, ei in zip(x, y, err))
        lo = " ".join(
            f"({xi:.4f},{yi - ei:.2f})" for xi, yi, ei in reversed(list(zip(x, y, err)))
        )
        L.append(
            f"\\addplot[{c}, fill={c}, fill opacity=0.3, "
            f"draw=none, forget plot] "
            f"coordinates {{{up} {lo}}} --cycle;"
        )
        coords = " ".join(f"({xi:.4f},{yi:.2f})" for xi, yi in zip(x, y))
        L.append(
            f"\\addplot[{c}, mark=*, mark size=1, "
            f"line width=1pt, forget plot] "
            f"coordinates {{{coords}}};"
        )

    if is_pct:
        L.append(
            f"\\addplot[gray, dashed, line width=0.5pt, "
            f"forget plot] coordinates "
            f"{{({_fx(xmin)},50) ({_fx(xmax)},50)}};"
        )
    for rl in ref_lines or []:
        L.append(
            f"\\addplot[gray, dashed, line width=0.5pt, "
            f"forget plot] coordinates "
            f"{{({xmin},{rl}) ({xmax},{rl})}};"
        )
    L.append(r"\end{axis}")
    L.append(r"\end{tikzpicture}")
    return "\n".join(L)


def latex_figure(
    curves_d,
    curves_n,
    metric,
    error_type,
    models,
    caption,
    label,
    ref_lines=None,
    ymin=None,
    ymax=None,
    noise_xlim=(None, None),
):
    d_tex = latex_panel(
        curves_d,
        DROPOUT_XLABEL["latex"],
        metric,
        error_type,
        models,
        ref_lines=ref_lines,
        ymin=ymin,
        ymax=ymax,
    )
    n_tex = latex_panel(
        curves_n,
        NOISE_XLABEL["latex"],
        metric,
        error_type,
        models,
        is_right=True,
        ref_lines=ref_lines,
        ymin=ymin,
        ymax=ymax,
        xlim=noise_xlim,
    )

    items = []
    # legend sorted alphabetically by the displayed model name (curves keep the MODELS order)
    for m in sorted(models, key=lambda m: MODEL_MACROS_LATEX[m][1].lower()):
        if m in curves_d or m in curves_n:
            c = MODEL_COLORS_LATEX[m]
            name = MODEL_MACROS_LATEX[m][0]
            items.append(
                f"\\tikz\\draw[{c}, thick, mark=*, "
                f"mark size=1.5] plot coordinates "
                f"{{(0,0) (0.4,0)}}; {{\\small {name}}}"
            )
    legend = "\\hspace{1em}\n".join(items)

    return (
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"{legend}\n"
        "\\\\[3pt]\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {d_tex}\n"
        f"\\end{{minipage}}\n"
        "\\hfill\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {n_tex}\n"
        f"\\end{{minipage}}\n"
        "\\vspace{-8pt}\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{fig:{label}}}\n"
        "\\end{figure}"
    )

## Main Experiments

Accuracy at detecting which sentence was perturbed, as a
function of perturbation strength. One curve per model.
Select a specific sentence length or pool across all six.

In [ ]:
def make_interactive(
    df_d, df_n, group_col, group_map, avail_metrics, section_label, ref_lines=None
):
    """Build a widget panel for one experiment section."""
    group_opts = [("all (pooled)", "all")] + [
        (label, val) for label, val in group_map.items()
    ]

    w_group = widgets.Dropdown(options=group_opts, value="all", description="Group:")
    w_metric = widgets.Dropdown(
        options=avail_metrics, value=avail_metrics[0], description="Metric:"
    )
    w_error = widgets.RadioButtons(
        options=["SE", "SD"],
        value="SE",
        description="Band:",
        layout=widgets.Layout(width="auto"),
    )
    w_models = {
        m: widgets.Checkbox(value=m in MODELS, description=m, indent=False)
        for m in ALL_MODELS
    }
    w_ymin = widgets.FloatText(
        value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
    )
    w_ymax = widgets.FloatText(
        value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
    )
    w_sxmin = widgets.FloatText(
        value=float("nan"), description="σ min:", layout=widgets.Layout(width="150px")
    )
    w_sxmax = widgets.FloatText(
        value=float("nan"), description="σ max:", layout=widgets.Layout(width="150px")
    )
    w_grid = widgets.Dropdown(
        options=["all", "coarse", "fine"], value="all", description="noise grid:"
    )
    out = widgets.Output()

    def redraw(*_):
        out.clear_output(wait=True)
        with out:
            gv = w_group.value
            metric = w_metric.value
            et = w_error.value
            models = [m for m in ALL_MODELS if w_models[m].value]
            ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
            ymax = w_ymax.value if not np.isnan(w_ymax.value) else None
            sxmin = w_sxmin.value if not np.isnan(w_sxmin.value) else None
            sxmax = w_sxmax.value if not np.isnan(w_sxmax.value) else None

            dd, dn = df_d, df_n
            # grid selector applies to both axes (zoom sweeps exist for dropout too)
            if w_grid.value != "all" and COL_NOISE_GRID in dd:
                dd = dd[dd[COL_NOISE_GRID] == w_grid.value]
            cd = compute_curves(
                dd, COL_DROPOUT, metric, group_col, gv, N_PER_RUN, models
            )
            if w_grid.value != "all":
                dn = dn[dn[COL_NOISE_GRID] == w_grid.value]
            cn = compute_curves(dn, COL_NOISE, metric, group_col, gv, N_PER_RUN, models)
            cn = clip_curves(
                cn, sxmin, sxmax
            )  # noise x-range (sigma min / max widgets)

            # --- matplotlib preview ---
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
            plot_panel(
                ax1,
                cd,
                DROPOUT_XLABEL["mpl"],
                metric,
                et,
                ymin,
                ymax,
                ref_lines=ref_lines,
            )
            plot_panel(
                ax2,
                cn,
                NOISE_XLABEL["mpl"],
                metric,
                et,
                ymin,
                ymax,
                is_right=True,
                ref_lines=ref_lines,
            )
            if sxmin is not None or sxmax is not None:
                ax2.set_xlim(left=sxmin, right=sxmax)

            handles, labels = ax1.get_legend_handles_labels()
            fig.legend(
                handles,
                labels,
                loc="upper center",
                ncol=max(len(models), 1),
                fontsize=10,
                bbox_to_anchor=(0.5, 1.08),
            )

            gl = (
                "pooled"
                if gv == "all"
                else (
                    _stem(gv) if "/" in str(gv) or "." in str(gv) else gv.split("/")[-1]
                )
            )
            fig.suptitle(
                f"{section_label} ({gl}, \u00b1{et})",
                fontsize=14,
                y=1.12,
            )
            fig.tight_layout()
            plt.show()

            # --- LaTeX ---
            caption = f"{section_label} accuracy ({gl})."
            safe = gl.replace("_", "-")
            label_tag = f"loc-{section_label.lower().split()[0]}-{safe}"
            if (
                section_label in PAPER_CAPTIONS
            ):  # paper figure: verbatim caption and label
                caption, label_tag = PAPER_CAPTIONS[section_label]
            print(
                latex_figure(
                    cd,
                    cn,
                    metric,
                    et,
                    models,
                    caption,
                    label_tag,
                    ref_lines=ref_lines,
                    ymin=ymin,
                    ymax=ymax,
                    noise_xlim=(sxmin, sxmax),
                )
            )

    for w in [w_group, w_metric, w_error, w_grid, w_ymin, w_ymax, w_sxmin, w_sxmax]:
        w.observe(redraw, names="value")
    for cb in w_models.values():
        cb.observe(redraw, names="value")

    display(
        widgets.HBox([w_group, w_metric, w_error, w_grid]),
        widgets.HBox([w_ymin, w_ymax, w_sxmin, w_sxmax]),
        widgets.HBox([w_models[m] for m in MODELS]),
        widgets.HBox([w_models[m] for m in STAGES]),
        out,
    )
    redraw()


make_interactive(df_main_d, df_main_n, COL_SENTENCES, sent_map, avail_main, "Main")

## Control Experiments

Controls use topic based prompts ("which sentence is about
animals?") instead of perturbation detection. There are
5 prompt pairs per (model, perturbation strength).

Extension noise data (finer granularity for llama, olmo32,
qwen32) is already merged into the noise DataFrame above.

In [ ]:
make_interactive(
    df_ctrl_d, df_ctrl_n, COL_PROMPT, prompt_map, avail_ctrl, "Control", ref_lines=[95]
)

## Per-model token count breakdown

Same main experiment data, but one model at a time with a
separate curve per sentence length. Color goes from light
pastel (short sentences) to dark (long sentences).

In [ ]:
SORTED_TOKS = sorted(sent_map.keys(), key=lambda s: int(s.replace("tok", "")))
TOK_COLORS = {
    "3tok": "#a8d8ea",  # light sky blue
    "7tok": "#f4b6c2",  # soft pink
    "11tok": "#b5e6a3",  # light green
    "15tok": "#f0c75e",  # warm gold
    "19tok": "#c4a4e0",  # lavender
    "23tok": "#e07b54",  # terracotta
}
TOK_COLORS_LATEX = {
    "3tok": "tokThree",
    "7tok": "tokSeven",
    "11tok": "tokEleven",
    "15tok": "tokFifteen",
    "19tok": "tokNineteen",
    "23tok": "tokTwentythree",
}
TOK_DEFINECOLORS = (
    "\\definecolor{tokThree}{HTML}{A8D8EA}\n"
    "\\definecolor{tokSeven}{HTML}{F4B6C2}\n"
    "\\definecolor{tokEleven}{HTML}{B5E6A3}\n"
    "\\definecolor{tokFifteen}{HTML}{F0C75E}\n"
    "\\definecolor{tokNineteen}{HTML}{C4A4E0}\n"
    "\\definecolor{tokTwentythree}{HTML}{E07B54}"
)


def _compute_tok_curves(df, x_col, metric, model):
    """Return {tok_label: (x, y, se, sd)} for a single model."""
    is_pct = "accuracy" in metric or "f1" in metric
    scale = 100 if is_pct else 1
    curves = {}
    sub_model = df[df[COL_MODEL] == model]
    if metric not in sub_model.columns or sub_model.empty:
        return curves
    for tok_label in SORTED_TOKS:
        sub = sub_model[sub_model[COL_SENTENCES] == sent_map[tok_label]]
        if sub.empty:
            continue
        grp = sub.groupby(x_col)[metric]
        avg = grp.mean().sort_index() * scale
        if avg.empty:
            continue
        if is_pct:
            p = avg / 100
            pq = p * (1 - p)
            n_total = grp.count() * N_PER_RUN
            se = np.sqrt(pq / n_total) * 100
            sd = np.sqrt(pq) * 100
        else:
            se = grp.sem().sort_index() * scale
            sd = grp.std().sort_index() * scale
        curves[tok_label] = (avg.index.values, avg.values, se.values, sd.values)
    return curves


def _plot_tok_panel(
    ax, curves, x_label, metric, error_type, ymin, ymax, is_right=False
):
    is_pct = "accuracy" in metric or "f1" in metric
    for tok_label in SORTED_TOKS:
        if tok_label not in curves:
            continue
        x, y, se, sd = curves[tok_label]
        err = se if error_type == "SE" else sd
        c = TOK_COLORS[tok_label]
        ax.plot(x, y, marker="o", markersize=3, linewidth=2, color=c, label=tok_label)
        ax.fill_between(x, y - err, y + err, alpha=0.2, color=c)

    if is_pct:
        ax.axhline(50, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel(x_label)
    if is_right:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Accuracy (%)" if is_pct else metric)
    ax.grid(True, alpha=0.3)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


def _latex_tok_panel(
    curves, x_label, metric, error_type, is_right=False, ymin=None, ymax=None
):
    is_pct = "accuracy" in metric or "f1" in metric
    all_x = (
        np.concatenate([c[0] for c in curves.values()]) if curves else np.array([0, 1])
    )
    xmin, xmax = float(all_x.min()), float(all_x.max())

    L = []
    L.append(r"\begin{tikzpicture}")
    L.append(r"\begin{axis}[")
    L.append("    height=7cm,")
    L.append("    width=\\linewidth,")
    L.append("    grid=major,")
    L.append(f"    xlabel={{{x_label}}},")
    if is_right:
        L.append("    ylabel={},")
        L.append("    yticklabels={},")
    else:
        ylabel = "Accuracy (\\%)" if is_pct else metric
        L.append(f"    ylabel={{{ylabel}}},")
    L.append(f"    xmin={xmin},")
    L.append(f"    xmax={xmax},")
    if ymin is not None and ymax is not None:
        L.append(f"    ymin={ymin}, ymax={ymax},")
    L.append("    legend entries={},")
    L.append("]")

    for tok_label in SORTED_TOKS:
        if tok_label not in curves:
            continue
        x, y, se, sd = curves[tok_label]
        err = se if error_type == "SE" else sd
        c = TOK_COLORS_LATEX[tok_label]

        up = " ".join(f"({xi:.4f},{yi + ei:.2f})" for xi, yi, ei in zip(x, y, err))
        lo = " ".join(
            f"({xi:.4f},{yi - ei:.2f})" for xi, yi, ei in reversed(list(zip(x, y, err)))
        )
        L.append(
            f"\\addplot[{c}, fill={c}, fill opacity=0.3, "
            f"draw=none, forget plot] "
            f"coordinates {{{up} {lo}}} --cycle;"
        )
        coords = " ".join(f"({xi:.4f},{yi:.2f})" for xi, yi in zip(x, y))
        L.append(
            f"\\addplot[{c}, mark=o, mark size=1, "
            f"line width=1pt, forget plot] "
            f"coordinates {{{coords}}};"
        )

    if is_pct:
        L.append(
            f"\\addplot[gray, dashed, line width=0.5pt, "
            f"forget plot] coordinates "
            f"{{({xmin},50) ({xmax},50)}};"
        )
    L.append(r"\end{axis}")
    L.append(r"\end{tikzpicture}")
    return "\n".join(L)


def _latex_tok_figure(
    curves_d, curves_n, metric, error_type, model, ymin=None, ymax=None
):
    d_tex = _latex_tok_panel(
        curves_d, DROPOUT_XLABEL["latex"], metric, error_type, ymin=ymin, ymax=ymax
    )
    n_tex = _latex_tok_panel(
        curves_n,
        NOISE_XLABEL["latex"],
        metric,
        error_type,
        is_right=True,
        ymin=ymin,
        ymax=ymax,
    )

    items = []
    for t in SORTED_TOKS:
        if t in curves_d or t in curves_n:
            c = TOK_COLORS_LATEX[t]
            items.append(
                f"\\tikz\\draw[{c}, thick, mark=o, "
                f"mark size=1.5] plot coordinates "
                f"{{(0,0) (0.4,0)}}; {t}"
            )
    legend = "\\hspace{1em}".join(items)
    err_tag = "SE" if error_type == "SE" else "SD"

    return (
        f"{TOK_DEFINECOLORS}\n"
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"{legend}\n"
        "\\\\[6pt]\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {d_tex}\n"
        f"\\end{{minipage}}\n"
        "\\hfill\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {n_tex}\n"
        f"\\end{{minipage}}\n"
        f"\\caption{{Token count breakdown for {MODEL_MACROS_LATEX[model][0]}. "
        f"Bands show $\\pm${err_tag}.}}\n"
        f"\\label{{fig:loc_tok_{model}}}\n"
        "\\end{figure}"
    )


def make_tok_interactive():
    w_model = widgets.Dropdown(
        options=ALL_MODELS, value=MODELS[0], description="Model:"
    )
    w_metric = widgets.Dropdown(
        options=avail_main, value=avail_main[0], description="Metric:"
    )
    w_error = widgets.RadioButtons(
        options=["SE", "SD"],
        value="SE",
        description="Band:",
        layout=widgets.Layout(width="auto"),
    )
    w_ymin = widgets.FloatText(
        value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
    )
    w_ymax = widgets.FloatText(
        value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
    )
    out = widgets.Output()

    def redraw(*_):
        out.clear_output(wait=True)
        with out:
            model = w_model.value
            metric = w_metric.value
            et = w_error.value
            ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
            ymax = w_ymax.value if not np.isnan(w_ymax.value) else None

            cd = _compute_tok_curves(df_main_d, COL_DROPOUT, metric, model)
            cn = _compute_tok_curves(df_main_n, COL_NOISE, metric, model)

            # --- matplotlib preview ---
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
            _plot_tok_panel(ax1, cd, DROPOUT_XLABEL["mpl"], metric, et, ymin, ymax)
            _plot_tok_panel(
                ax2, cn, NOISE_XLABEL["mpl"], metric, et, ymin, ymax, is_right=True
            )

            handles, labels = ax1.get_legend_handles_labels()
            fig.legend(
                handles,
                labels,
                loc="upper center",
                ncol=len(SORTED_TOKS),
                fontsize=10,
                bbox_to_anchor=(0.5, 1.08),
            )
            fig.suptitle(
                f"Token count breakdown \u2014 {model} (\u00b1{et})",
                fontsize=14,
                y=1.12,
            )
            fig.tight_layout()
            plt.show()

            # --- LaTeX ---
            print(_latex_tok_figure(cd, cn, metric, et, model, ymin=ymin, ymax=ymax))

    for w in [w_model, w_metric, w_error, w_ymin, w_ymax]:
        w.observe(redraw, names="value")

    display(
        widgets.HBox([w_model, w_metric, w_error]),
        widgets.HBox([w_ymin, w_ymax]),
        out,
    )
    redraw()


make_tok_interactive()

## Bounds overview

One row per model: detection accuracy (`accuracy_primary`; OLMo uses the constrained rule because it
answers "neither" at low strengths) and control content accuracy against the strength, dropout on the
left and noise on the right, all grids pooled. Vertical lines: **min** = last strength inside the
chance band before a sustained rise (two consecutive points above $0.5 + z\,\mathrm{SE}$), **max** = last
strength whose mean content accuracy is still $\ge 95\%$. The table below lists them.


In [ ]:
BOUND_Z = 2.0
BOUND_METRIC = {m: "accuracy_primary" for m in ALL_MODELS}
for _m in ["olmo3_32b"] + STAGES:
    BOUND_METRIC[_m] = "accuracy_constrained_primary"
_band = BOUND_Z * np.sqrt(0.25 / N_PER_RUN)


def _curve(df, model, x_col, metric):
    sub = df[df[COL_MODEL] == model]
    if sub.empty or metric not in sub.columns:
        return pd.Series(dtype=float)
    return sub.groupby(x_col)[metric].mean().sort_index()


def bound_min(acc):
    """Strength before the first two consecutive points above the chance band (None if never)."""
    up = (acc > 0.5 + _band).tolist()
    idx = list(acc.index)
    for i in range(len(up) - 1):
        if up[i] and up[i + 1]:
            return idx[i - 1] if i > 0 else None
    return None


def bound_max(content, thr=0.95):
    """Strength before the first mean content accuracy below `thr` (None if never)."""
    below = (content < thr).tolist()
    idx = list(content.index)
    for i, b in enumerate(below):
        if b:
            return idx[i - 1] if i > 0 else None
    return None


_present = [m for m in ALL_MODELS if (df_main_d[COL_MODEL] == m).any()]
rows = []
fig, axes = plt.subplots(
    len(_present), 2, figsize=(16, 3.2 * len(_present)), squeeze=False
)
for r, model in enumerate(_present):
    metric = BOUND_METRIC[model]
    row = {"model": model, "metric": metric}
    for c, (pert, x_col, df_m, df_c, xlabel) in enumerate(
        [
            ("dropout", COL_DROPOUT, df_main_d, df_ctrl_d, DROPOUT_XLABEL["mpl"]),
            ("noise", COL_NOISE, df_main_n, df_ctrl_n, NOISE_XLABEL["mpl"]),
        ]
    ):
        acc = _curve(df_m, model, x_col, metric)
        content = _curve(df_c, model, x_col, f"content_{metric}")
        lo, hi = bound_min(acc), bound_max(content)
        row[f"{pert}_min"], row[f"{pert}_max"] = lo, hi
        ax = axes[r, c]
        ax.plot(
            acc.index,
            acc.values,
            "o-",
            ms=3,
            color=MODEL_COLORS_MPL[model],
            label="detection",
        )
        ax.plot(
            content.index,
            content.values,
            "s--",
            ms=3,
            color="gray",
            label="control content",
        )
        ax.axhspan(0.5 - _band, 0.5 + _band, color="k", alpha=0.08)
        ax.axhline(0.95, color="gray", ls=":", lw=1)
        for v, ls in ((lo, "-"), (hi, "--")):
            if v is not None:
                ax.axvline(v, color="red", ls=ls, lw=1)
        ax.set_title(f"{model} / {pert}   min={lo}  max={hi}", fontsize=10)
        ax.set_xlabel(xlabel)
        ax.set_ylim(0, 1.02)
        if c == 0:
            ax.set_ylabel("accuracy")
        if r == 0 and c == 0:
            ax.legend(fontsize=8, loc="lower right")
    rows.append(row)
fig.tight_layout()
plt.show()

bounds_table = pd.DataFrame(rows).set_index("model")
display(bounds_table)

## Yes/no detection: "was a perturbation applied?"

Every sample is perturbed (dropout or noise) and the model is asked whether a perturbation was applied, with the
options YES and NO in random order. `accuracy_primary` is then P(YES | perturbation); strength 0 is the run with
nothing applied. Sweeps: manifest keys `yesno_dropout` / `yesno_noise` (every model) and `olmo_stages` →
`yesno_stage_{base,sft,dpo}_*` (RLVR = the final `olmo3_32b`). Sweeps still running are shown but not cached.
The toggle `question` shows the controls (manifest keys `yesno_ctrl_*`): a neutral prompt (read a sentence, answer a
question) whose question has the answer NO, the answer YES, or no answer (e.g. a coin flip); the metric is still P(YES).

The OLMo stage runs use the RLVR range (the olmo3_32b row of Table 1); the toggle `stage range = own` shows them in % of each stage's own range, from the bounds of the cell above.

In [ ]:
# --- Yes/no detection: interactive view ---
# One table with every run of the yes/no sweeps; in a run the applied perturbation is the YES class.
YESNO_RUNS = 12  # strength 0 + the 11 bins of the model's range; unfinished sweeps are shown but not cached
YESNO_QUESTIONS = {  # toggle label -> prompts/turns of the runs
    "perturbation applied?": "classification/perturbed",
    "answer NO": "classification/perturbed_ctrl_no",
    "answer YES": "classification/perturbed_ctrl_yes",
    "no answer": "classification/perturbed_ctrl_none",
}

_stage_names = {
    "base": "Olmo3.1-32B Base",
    "sft": "Olmo3.1-32B SFT",
    "dpo": "Olmo3.1-32B DPO",
}
_yesno_sweeps = {}  # label -> {pert: {sweep id: number of questions}}
_yesno_stage = {name: f"olmo3_32b_instruct_{s}" for s, name in _stage_names.items()}
for model, ids in _manifest.items():
    if isinstance(ids, dict) and "yesno_dropout" in ids:
        label = model + (" (RLVR)" if model == "olmo3_32b" else "")
        _yesno_sweeps[label] = {
            p: {ids[f"yesno_{p}"]: 1, ids[f"yesno_ctrl_{p}"]: 3}
            for p in ("dropout", "noise")
        }
for stage, name in _stage_names.items():
    _yesno_sweeps[name] = {
        p: {
            _manifest["olmo_stages"][f"yesno_stage_{stage}_{p}"]: 1,
            _manifest["olmo_stages"][f"yesno_ctrl_stage_{stage}_{p}"]: 3,
        }
        for p in ("dropout", "noise")
    }

YESNO_METRICS = {  # label -> (column for the YES class of a run, its SE column or None, scale)
    "P(answers YES), exact token (%)": ("accuracy_primary", "accuracy_se_primary", 100),
    "P(answers YES), only A vs B (%)": (
        "accuracy_constrained_primary",
        "accuracy_se_constrained_primary",
        100,
    ),
    "answers with a letter (%)": ("argmax_is_label_rate", None, 100),
    "mean probability of YES (%)": (
        "mean_primary_prob_{yes}",
        "se_primary_prob_{yes}",
        100,
    ),
    "mean probability of NO (%)": (
        "mean_primary_prob_{no}",
        "se_primary_prob_{no}",
        100,
    ),
    "logit YES - logit NO": ("mean_logit_diff_{yes}", "se_logit_diff_{yes}", 1),
    "mean probability of A (%)": ("position_bias_primary_prob_A", None, 100),
    "mean probability of B (%)": ("position_bias_primary_prob_B", None, 100),
    "entropy (nats)": ("entropy_mean", "entropy_se", 1),
}

_rows = []
for label, ids in _yesno_sweeps.items():
    for pert, sids in ids.items():
        col = COL_DROPOUT if pert == "dropout" else COL_NOISE
        yes, no = ("dropout", "noise") if pert == "dropout" else ("noise", "dropout")
        for sid, n_questions in sids.items():
            df = load_sweep(
                sid,
                _manifest["project"],
                require_version=2,
                min_rows=YESNO_RUNS * n_questions,
            )
            if df.empty:
                continue
            for question, dq in df.groupby(COL_PROMPT):
                dq = dq.sort_values(col).reset_index(drop=True)
                for i, r in dq.iterrows():
                    row = {
                        "model": label,
                        "question": question,
                        "pert": pert,
                        "strength": float(r[col]),
                        "bin": i,
                        "runs": len(dq),
                    }
                    for name, (c, se, scale) in YESNO_METRICS.items():
                        c, se = (
                            c.format(yes=yes, no=no),
                            se.format(yes=yes, no=no) if se else None,
                        )
                        row[name] = r.get(c, np.nan) * scale
                        row[name + " SE"] = r.get(se, np.nan) * scale if se else np.nan
                    _rows.append(row)
yesno = pd.DataFrame(_rows)

yn_models = widgets.SelectMultiple(
    options=list(_yesno_sweeps),
    value=list(_yesno_sweeps)[:8],
    rows=11,
    description="models",
)
yn_pert = widgets.ToggleButtons(
    options=["dropout", "noise", "both"], value="both", description="perturbation"
)
yn_question = widgets.ToggleButtons(
    options=list(YESNO_QUESTIONS), value="perturbation applied?", description="question"
)
yn_metric = widgets.Dropdown(
    options=list(YESNO_METRICS),
    value="P(answers YES), exact token (%)",
    description="metric",
    layout=widgets.Layout(width="420px"),
)
yn_x = widgets.ToggleButtons(
    options=["bin in range", "strength"], value="bin in range", description="x axis"
)
yn_range = widgets.ToggleButtons(
    options=["RLVR", "own"],
    value="RLVR",
    description="stage range",
    tooltip="the stage runs use the RLVR range; 'own' shows them in % of the stage's range (cell 12)",
)
yn_se = widgets.Checkbox(value=True, description="error bars (SE)")
yn_table = widgets.Checkbox(value=False, description="show table")
yn_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
yn_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)


def _yesno_x(d, m, p, x, rng):
    """Bin mode: 0 = none, 1 + i = 10 i % of the range (the stage's own range if rng == "own")."""
    if x != "bin in range":
        return d["strength"]
    if rng == "own" and m in _yesno_stage:
        lo, hi = bounds_table.loc[_yesno_stage[m], [f"{p}_min", f"{p}_max"]]
        if lo is not None and hi is not None and hi > lo:
            s = d["strength"]
            return (1 + 10 * (s - lo) / (hi - lo)).where(s > 0, 0)
    return d["bin"]


def _yesno_plot(models, question, pert, metric, x, rng, se, table, ymin, ymax):
    perts = ["dropout", "noise"] if pert == "both" else [pert]
    q = yesno[yesno.question == YESNO_QUESTIONS[question]]
    fig, axes = plt.subplots(
        1, len(perts), figsize=(5.5 * len(perts), 3.8), squeeze=False, sharey=True
    )
    for ax, p in zip(axes[0], perts):
        top = YESNO_RUNS - 1
        for m in models:
            d = q[(q.model == m) & (q.pert == p)]
            xs = _yesno_x(d, m, p, x, rng)
            top = max(top, int(np.ceil(xs.max()))) if len(xs) else top
            base = m.split(" (")[0] if m.endswith("(RLVR)") else _yesno_stage.get(m, m)
            color = MODEL_COLORS_MPL.get(base)
            (line,) = ax.plot(xs, d[metric], marker="o", ms=3, color=color, label=m)
            if se and d[metric + " SE"].notna().any():
                ax.fill_between(
                    xs,
                    d[metric] - d[metric + " SE"],
                    d[metric] + d[metric + " SE"],
                    color=line.get_color(),
                    alpha=0.2,
                )
        if x == "bin in range":
            ax.set_xticks(range(top + 1), ["none"] + [f"{10 * i}%" for i in range(top)])
            ax.tick_params(axis="x", labelsize=7)
            xl = "strength (% of the range)"
        else:
            xl = "dropout rate $p$" if p == "dropout" else r"noise SD $\sigma$"
        ax.set(title=p, xlabel=xl)
    axes[0][0].set_ylabel(metric)
    axes[0][0].set_ylim(
        None if np.isnan(ymin) else ymin, None if np.isnan(ymax) else ymax
    )
    axes[0][-1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.tight_layout()
    plt.show()
    missing = sorted(
        {
            m
            for m in models
            for p in perts
            if (q[(q.model == m) & (q.pert == p)]["runs"] < YESNO_RUNS).any()
            or not ((q.model == m) & (q.pert == p)).any()
        }
    )
    if missing:
        print("still running (not all 12 strengths finished):", ", ".join(missing))
    if table:
        t = q[q.model.isin(models) & q.pert.isin(perts)]
        display(
            t.pivot_table(index=["pert", "bin"], columns="model", values=metric).round(
                2
            )
        )


display(
    widgets.VBox(
        [
            widgets.HBox(
                [
                    yn_models,
                    widgets.VBox(
                        [
                            yn_question,
                            yn_pert,
                            yn_metric,
                            yn_x,
                            yn_range,
                            widgets.HBox([yn_se, yn_table]),
                            widgets.HBox([yn_ymin, yn_ymax]),
                        ]
                    ),
                ]
            ),
            widgets.interactive_output(
                _yesno_plot,
                {
                    "models": yn_models,
                    "question": yn_question,
                    "pert": yn_pert,
                    "metric": yn_metric,
                    "x": yn_x,
                    "rng": yn_range,
                    "se": yn_se,
                    "table": yn_table,
                    "ymin": yn_ymin,
                    "ymax": yn_ymax,
                },
            ),
        ]
    )
)